# Tableau Dashboard Aggregate Exports

This notebook creates Tableau-ready aggregate CSV files for the EHR readmissions portfolio project. It uses existing aggregate project outputs only and does not export raw patient-level records.

## Purpose

The Tableau dashboard should be built manually from clean, dashboard-friendly CSV extracts. These files support KPI cards, subgroup readmission views, utilization comparisons, and data quality reporting using synthetic Synthea EHR data.

In [ ]:
from pathlib import Path
import re
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLEAU_DIR = PROJECT_ROOT / "tableau_dashboard"
BI_DIR = PROJECT_ROOT / "outputs" / "bi"
ANALYSIS_DIR = PROJECT_ROOT / "outputs" / "analysis"
VALIDATION_DIR = PROJECT_ROOT / "outputs" / "validation"
TABLEAU_DIR.mkdir(exist_ok=True)

def parse_count_percent(value):
    match = re.match(r"\s*(\d+)\s*\(([-0-9.]+)%\)", str(value))
    if not match:
        return None, None
    return int(match.group(1)), float(match.group(2))

def pct(numer, denom):
    return round(100 * numer / denom, 1) if denom else 0.0

## Load Existing Aggregate Outputs

These inputs come from the SQL, validation, BI, and analysis workflow already present in the project.

In [ ]:
cohort = pd.read_csv(BI_DIR / "cohort_summary_table.csv").iloc[0]
readmission = pd.read_csv(BI_DIR / "readmission_kpi_table.csv").iloc[0]
followup = pd.read_csv(BI_DIR / "followup_timing_table.csv")
ed = pd.read_csv(BI_DIR / "ed_revisit_table.csv").iloc[0]
demo = pd.read_csv(BI_DIR / "demographic_utilization_summary_table.csv")
table1 = pd.read_csv(ANALYSIS_DIR / "table1_baseline_characteristics.csv")
table1["variable_filled"] = table1["variable"].replace("", pd.NA).ffill()
attrition = pd.read_csv(VALIDATION_DIR / "cohort_attrition_counts.csv")
index_check = pd.read_csv(VALIDATION_DIR / "index_encounter_check.csv").iloc[0]
date_checks = pd.read_csv(VALIDATION_DIR / "date_validity_checks.csv")
missingness = pd.read_csv(VALIDATION_DIR / "missingness_report.csv")
readmit_timing = pd.read_csv(VALIDATION_DIR / "readmission_timing_validation.csv").iloc[0]
followup_timing = pd.read_csv(VALIDATION_DIR / "outpatient_followup_timing_validation.csv").iloc[0]

## Create Tableau-Ready CSV Files

Each export is aggregated and designed for direct connection in Tableau. Running this cell refreshes all CSV files in `tableau_dashboard/`.

In [ ]:

cohort_n = int(cohort["cohort_patients"])
followup_30 = followup.loc[followup["followup_window_days"] == 30].iloc[0]

kpi_summary = pd.DataFrame([
    {"kpi_name": "Final analytic cohort", "kpi_category": "Cohort", "count": cohort_n, "denominator": cohort_n, "percent": 100.0, "display_value": f"{cohort_n:,} patients", "interpretation_note": "Adult patients with one first eligible index inpatient encounter. Synthetic data only."},
    {"kpi_name": "30-day inpatient readmission", "kpi_category": "Outcome", "count": int(readmission["readmitted_30d_count"]), "denominator": int(readmission["cohort_patients"]), "percent": float(readmission["readmission_rate_30d_percent"]), "display_value": f"{float(readmission['readmission_rate_30d_percent']):.1f}%", "interpretation_note": "All-cause inpatient readmission within 30 days of index discharge. Descriptive only."},
    {"kpi_name": "30-day outpatient follow-up", "kpi_category": "Utilization", "count": int(followup_30["followup_count"]), "denominator": cohort_n, "percent": float(followup_30["followup_percent"]), "display_value": f"{float(followup_30['followup_percent']):.1f}%", "interpretation_note": "Outpatient or ambulatory follow-up within 30 days. Not interpreted causally."},
    {"kpi_name": "30-day ED revisit", "kpi_category": "Utilization", "count": int(ed["ed_revisit_30d_count"]), "denominator": int(ed["cohort_patients"]), "percent": float(ed["ed_revisit_30d_percent"]), "display_value": f"{float(ed['ed_revisit_30d_percent']):.1f}%", "interpretation_note": "ED revisit within 30 days. Rare event in this synthetic cohort."},
    {"kpi_name": "Any post-discharge encounter within 30 days", "kpi_category": "Utilization", "count": int(ed["any_postdischarge_encounter_30d_count"]), "denominator": int(ed["cohort_patients"]), "percent": float(ed["any_postdischarge_encounter_30d_percent"]), "display_value": f"{float(ed['any_postdischarge_encounter_30d_percent']):.1f}%", "interpretation_note": "Any observed post-discharge encounter within 30 days."},
])
kpi_summary.to_csv(TABLEAU_DIR / "kpi_summary.csv", index=False)

age = demo.loc[demo["summary_domain"] == "age_group"].copy()
age["sort_order"] = age["category"].map({"Under 65": 1, "65+": 2}).fillna(99).astype(int)
age = age.sort_values("sort_order")
age = age[["category", "patient_count", "patient_percent_within_domain", "readmitted_30d_count", "readmission_rate_30d_percent", "outpatient_followup_30d_count", "outpatient_followup_30d_percent", "ed_revisit_30d_count", "ed_revisit_30d_percent", "mean_length_of_stay_days", "mean_prior_encounters_12mo", "mean_chronic_condition_count", "sort_order"]].rename(columns={"category": "age_group"})
age["interpretation_note"] = "Synthetic subgroup summary; descriptive only. Small readmission counts should not be overinterpreted."
age.to_csv(TABLEAU_DIR / "readmission_by_age_group.csv", index=False)

condition_labels = {"Diabetes flag": "Diabetes", "Hypertension flag": "Hypertension", "CKD flag": "Chronic kidney disease", "COPD flag": "COPD"}
condition_rows = []
for variable, label in condition_labels.items():
    p_value_row = table1.loc[table1["variable"] == variable, "p_value"]
    p_value = p_value_row.iloc[0] if not p_value_row.empty else ""
    row = table1.loc[(table1["variable_filled"] == variable) & (table1["level"].astype(str) == "1")].iloc[0]
    patient_count, patient_percent = parse_count_percent(row["overall"])
    not_readmitted_count, _ = parse_count_percent(row["not_readmitted"])
    readmitted_count, _ = parse_count_percent(row["readmitted"])
    condition_rows.append({"condition_group": label, "patient_count": patient_count, "patient_percent": patient_percent, "readmitted_30d_count": readmitted_count, "not_readmitted_count": not_readmitted_count, "readmission_rate_30d_percent": pct(readmitted_count, patient_count), "p_value": p_value, "interpretation_note": "Derived from simplified Synthea condition flags in aggregate Table 1; descriptive only."})
condition = pd.DataFrame(condition_rows).sort_values("condition_group")
condition.to_csv(TABLEAU_DIR / "readmission_by_condition_group.csv", index=False)

util_rows = []
for _, row in followup.iterrows():
    util_rows.append({"measure_name": f"Outpatient follow-up {row['followup_window']}", "measure_group": "Outpatient follow-up", "window_days": int(row["followup_window_days"]), "count": int(row["followup_count"]), "denominator": cohort_n, "percent": float(row["followup_percent"]), "mean_days_to_event": float(row["mean_days_to_first_followup"]), "interpretation_note": "Cumulative outpatient or ambulatory follow-up window after discharge."})
util_rows.extend([
    {"measure_name": "30-day ED revisit", "measure_group": "ED revisit", "window_days": 30, "count": int(ed["ed_revisit_30d_count"]), "denominator": int(ed["cohort_patients"]), "percent": float(ed["ed_revisit_30d_percent"]), "mean_days_to_event": "", "interpretation_note": "ED revisit within 30 days; secondary utilization measure."},
    {"measure_name": "30-day inpatient readmission", "measure_group": "Readmission", "window_days": 30, "count": int(readmission["readmitted_30d_count"]), "denominator": int(readmission["cohort_patients"]), "percent": float(readmission["readmission_rate_30d_percent"]), "mean_days_to_event": float(readmission["mean_days_to_readmission"]), "interpretation_note": "Primary outcome; all-cause inpatient readmission within 30 days."},
    {"measure_name": "Any post-discharge encounter within 30 days", "measure_group": "Any encounter", "window_days": 30, "count": int(ed["any_postdischarge_encounter_30d_count"]), "denominator": int(ed["cohort_patients"]), "percent": float(ed["any_postdischarge_encounter_30d_percent"]), "mean_days_to_event": "", "interpretation_note": "Any observed post-discharge encounter within 30 days."},
])
util = pd.DataFrame(util_rows)
util.to_csv(TABLEAU_DIR / "utilization_summary.csv", index=False)

attr_lookup = dict(zip(attrition["step"], attrition["record_count"]))
final_date = date_checks.loc[date_checks["dataset"] == "final_analysis_dataset"].iloc[0]
invalid_final_dates = int(final_date["missing_start"] + final_date["missing_stop"] + final_date["invalid_start"] + final_date["invalid_stop"] + final_date["stop_before_start"])
quality_rows = [
    {"quality_domain": "Cohort construction", "check_name": "Final analysis dataset rows", "value": int(attr_lookup.get("final_analysis_dataset_rows", cohort_n)), "denominator": cohort_n, "percent": 100.0, "status": "Pass", "interpretation_note": "Final aggregate cohort available for dashboard reporting."},
    {"quality_domain": "Index encounter", "check_name": "One index encounter per patient", "value": int(index_check["index_rows"] == index_check["distinct_patients"]), "denominator": 1, "percent": 100.0 if index_check["index_rows"] == index_check["distinct_patients"] else 0.0, "status": "Pass" if index_check["index_rows"] == index_check["distinct_patients"] else "Review", "interpretation_note": f"{int(index_check['index_rows'])} index rows and {int(index_check['distinct_patients'])} distinct patients."},
    {"quality_domain": "Index encounter", "check_name": "Duplicate patient index rows", "value": int(index_check["duplicate_patient_index_rows"]), "denominator": int(index_check["index_rows"]), "percent": pct(int(index_check["duplicate_patient_index_rows"]), int(index_check["index_rows"])), "status": "Pass" if int(index_check["duplicate_patient_index_rows"]) == 0 else "Review", "interpretation_note": "Duplicate patient index rows should be zero."},
    {"quality_domain": "Date validity", "check_name": "Invalid or missing final encounter dates", "value": invalid_final_dates, "denominator": int(final_date["row_count"]), "percent": pct(invalid_final_dates, int(final_date["row_count"])), "status": "Pass" if invalid_final_dates == 0 else "Review", "interpretation_note": "Missing, invalid, or temporally inconsistent final analysis dates."},
    {"quality_domain": "Outcome derivation", "check_name": "Missing readmission outcome rows", "value": int(readmit_timing["missing_readmission_outcome_rows"]), "denominator": int(readmit_timing["final_rows"]), "percent": pct(int(readmit_timing["missing_readmission_outcome_rows"]), int(readmit_timing["final_rows"])), "status": "Pass" if int(readmit_timing["missing_readmission_outcome_rows"]) == 0 else "Review", "interpretation_note": "Primary outcome should be populated for all final analysis rows."},
    {"quality_domain": "Follow-up timing", "check_name": "Non-monotonic follow-up window rows", "value": int(followup_timing["non_monotonic_followup_window_rows"]), "denominator": int(followup_timing["final_rows"]), "percent": pct(int(followup_timing["non_monotonic_followup_window_rows"]), int(followup_timing["final_rows"])), "status": "Pass" if int(followup_timing["non_monotonic_followup_window_rows"]) == 0 else "Review", "interpretation_note": "7/14/30-day outpatient follow-up flags should be monotonic."},
]
for _, row in missingness.head(8).iterrows():
    quality_rows.append({"quality_domain": "Missingness", "check_name": f"Missingness: {row['variable']}", "value": int(row["missing_count"]), "denominator": int(row["row_count"]), "percent": float(row["missing_percent"]), "status": "Expected" if row["variable"] in ["days_to_readmission", "readmission_encounter_id", "days_to_first_outpatient_followup", "days_to_first_postdischarge_encounter"] else "Review", "interpretation_note": "Timing fields are missing when the corresponding event is not observed; other variables require review."})
quality = pd.DataFrame(quality_rows)
quality.to_csv(TABLEAU_DIR / "data_quality_summary.csv", index=False)

for path in sorted(TABLEAU_DIR.glob("*.csv")):
    print(path.relative_to(PROJECT_ROOT))
